<a href="https://colab.research.google.com/github/kph4br/ds2002-fa26/blob/main/notebooks/01-foundations/2026-09-30-JSON-to-Tidy-Tables-Studio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · JSON to Tidy Tables

**Studio — 2026-09-30 · Fall 2026**  
**Class time:** 45 minutes

---

## One response, straight off the wire

Monday you took apart a payload that had already been laid out for you, indented and readable. Real responses do not arrive that way. Whitespace costs bandwidth, so APIs ship their JSON with every space squeezed out of it.

Here is a portfolio response from a market-data service, exactly as it came down. Run the cell and try to read it.

In [1]:
RAW = '{"as_of":"2026-09-30","currency":"USD","source":"teaching-fixture","portfolio":{"name":"Cavalier Growth","holdings":[{"ticker":"AAPL","company":{"name":"Apple Inc.","sector":"Technology"},"shares":120,"prices":[{"date":"2026-09-28","close":231.4,"volume":52300000},{"date":"2026-09-29","close":229.85,"volume":48100000},{"date":"2026-09-30","close":233.1,"volume":55600000}]},{"ticker":"MSFT","company":{"name":"Microsoft Corp.","sector":"Technology"},"shares":45,"prices":[{"date":"2026-09-28","close":502.1,"volume":22100000},{"date":"2026-09-29","close":"498.75","volume":0},{"date":"2026-09-30","close":505.3,"volume":24800000}]},{"ticker":"KO","company":{"name":"Coca-Cola Co."},"shares":300,"prices":[{"date":"2026-09-28","close":71.25,"volume":12400000},{"date":"2026-09-29","close":null,"volume":13000000},{"date":"2026-09-30","close":70.9,"volume":11800000}]}]}}'
print('characters:', len(RAW))
print(RAW[:160], '...')

characters: 871
{"as_of":"2026-09-30","currency":"USD","source":"teaching-fixture","portfolio":{"name":"Cavalier Growth","holdings":[{"ticker":"AAPL","company":{"name":"Apple I ...


Eight hundred and seventy-one characters on one line. You cannot read that, and neither can anyone else. That is fine. **Nobody reads raw JSON.** You parse it, then you ask Python to lay it out for you.

Notice what you are holding right now: `RAW` is a *string*. It has a length. You could slice it. There are no keys to look up yet, because text does not have keys.

### Step 1 — parse it, then make it readable

Two different jobs, and people confuse them constantly. `json.loads` turns text into Python objects. `json.dumps` with `indent` turns Python objects back into text that a human can scan. You are about to do both, in that order, and end up looking at the same data you started with — just formatted.

In [2]:
import json, pandas as pd

data = json.loads(RAW)
print(type(RAW), '->', type(data))
print(json.dumps(data, indent=2)[:700], '...')

<class 'str'> -> <class 'dict'>
{
  "as_of": "2026-09-30",
  "currency": "USD",
  "source": "teaching-fixture",
  "portfolio": {
    "name": "Cavalier Growth",
    "holdings": [
      {
        "ticker": "AAPL",
        "company": {
          "name": "Apple Inc.",
          "sector": "Technology"
        },
        "shares": 120,
        "prices": [
          {
            "date": "2026-09-28",
            "close": 231.4,
            "volume": 52300000
          },
          {
            "date": "2026-09-29",
            "close": 229.85,
            "volume": 48100000
          },
          {
            "date": "2026-09-30",
            "close": 233.1,
            "volume": 55600000
          }
        ]
      },
      { ...


Same information, 871 characters versus something much longer, and now you can see the shape. `indent=2` is the first thing to reach for when a payload lands in your lap.

**TODO:** in one sentence, why is it wrong to say the API "sent you a dictionary"?

**Answer:** The API sent you a string of text, not a dictionary. json.loads is what turned that text into a Python dictionary on my end.

### Step 2 — map the shape before you touch it

Never normalize a payload you have not walked. You need two numbers before you go any further: how many holdings there are, and how many price rows you should end up with.

The first line is done for you — `.keys()` gives you an object's names when you do not already know them. The counting is yours.

**TODO:** print the portfolio name, the number of holdings, and the total number of price records across all holdings.

In [3]:
print('top-level keys:', list(data.keys()))

# TODO: the portfolio name
# TODO: how many holdings?
# TODO: how many price records in total? (len() on each holding's prices)

holdings = data['portfolio']['holdings']
print('portfolio:', data['portfolio']['name'])
print('holdings count:', len(holdings))
print('price rows:', sum(len(h['prices']) for h in holdings))

top-level keys: ['as_of', 'currency', 'source', 'portfolio']
portfolio: Cavalier Growth
holdings count: 3
price rows: 9


**TODO:** write your two numbers down here, because every assertion below checks against them.

- holdings: 3
- price rows: 9
- one row of the table you are about to build is one ticker on one day.

### Step 3 — trace a path, and hit the wall

A path is the sequence of names and positions that gets you to one value. Reading a path is the skill; the brackets are just typing.

**TODO:** print Apple's sector, then try to print Coca-Cola's the same way. One of those raises an exception. Catch it and read what it says. Then rewrite the line so it checks whether the name is there first and prints `'unknown'` when it is not — the `'pm25' not in r` test from Monday's missing/null/empty/zero table, pointed at `'sector'`.

In [4]:
# TODO: data['portfolio']['holdings'][0]['company']['sector']
print(data['portfolio']['holdings'][0]['company']['sector'])

# TODO: now try the same thing for KO -- what happens, and why?
try:
    print(data['portfolio']['holdings'][2]['company']['sector'])
except KeyError as e:
    print('KeyError:', e)

# TODO: rewrite it using an `in` check so a missing sector prints 'unknown'
ko = data['portfolio']['holdings'][2]['company']
print(ko['sector'] if 'sector' in ko else 'unknown')

Technology
KeyError: 'sector'
unknown


### Step 4 — the four checks

It decoded and it parsed, so checks 1 and 2 passed the moment `json.loads` returned. Check 3 asks whether the data matches what you were promised, and check 4 asks whether it means what you think.

This payload has **four** planted problems. Three of them are in the price records and one is in a company object. None of them will raise an error.

**TODO:** walk every price record and report, with the ticker and date for each:

1. a `close` that came through as `None`
2. a `close` that is a *string* when every other one is a number
3. a `volume` of exactly `0`
4. a holding whose company object is missing a field the others have

Then answer the one that matters: **which of those four are genuinely missing data, and which are real measurements?** They do not get handled the same way.

In [5]:
# TODO: loop over holdings, then over prices, and report the four problems
#
# Hint for #2: isinstance(value, str)
# Hint for #3: write it as `volume == 0`, not `if not volume` -- ask yourself why
for h in data['portfolio']['holdings']:
    if 'sector' not in h['company']:
        print(f"{h['ticker']}: company missing 'sector'")
    for p in h['prices']:
        if p['close'] is None:
            print(f"{h['ticker']} {p['date']}: close is None")
        if isinstance(p['close'], str):
            print(f"{h['ticker']} {p['date']}: close is a string {p['close']!r}")
        if p['volume'] == 0:
            print(f"{h['ticker']} {p['date']}: volume is 0")

MSFT 2026-09-29: close is a string '498.75'
MSFT 2026-09-29: volume is 0
KO: company missing 'sector'
KO 2026-09-29: close is None


**TODO:** your answer to the last question: Four issues
1) KO has no setor
2) KO's close on 09-29 is None
3) MSFT's close on 09-29 is the string '498.75'
4) MSFT's volume on 09-29 is 0

The None close and the missing sector are missing data. The string '498.75' is just a measurement stored as the wrong data type, so you can just coerce it instead of drop it. The 0 volume is a recorded value, not an absence I hope, so I'm not treating it as missing by default.

### Step 5 — flatten to one row per ticker per day

`record_path` is the list you want one row from. `meta` is everything above it that should be copied down onto each of those rows. `sector` only exists on two of the three companies, so you will need `errors='ignore'` to keep `json_normalize` from refusing.

*Expected: 9 rows.*

In [6]:
# TODO: px = pd.json_normalize(holdings, record_path=..., meta=[...], sep='.',
#                              errors='ignore')
# TODO: assert len(px) == 9
# TODO: print px.dtypes and find the column that is the wrong type
px = pd.json_normalize(holdings, record_path='prices',
                       meta=['ticker', 'shares', ['company', 'name'], ['company', 'sector']],
                       sep='.', errors='ignore')
assert len(px) == 9
print('length:', len(px))
print(px.dtypes)

length: 9
date              object
close             object
volume             int64
ticker            object
shares            object
company.name      object
company.sector    object
dtype: object


### Step 6 — fix the type drift

One string in a column of numbers is enough to make the whole column text, and a text column will not average. `pd.to_numeric(..., errors='coerce')` converts what it can and turns the rest into `NaN`.

**TODO:** convert `close` to a number, then confirm the dtype changed and count how many nulls you are left with. Say out loud where that null came from — it is not the string.

In [7]:
# TODO: px['close'] = pd.to_numeric(...)
# TODO: check the dtype and the null count
px['close'] = pd.to_numeric(px['close'], errors='coerce')
px['shares'] = px['shares'].astype(int)
print(px['close'].dtype, px['close'].isna().sum())

float64 1


### Step 7 — answer the questions

**TODO:** produce each of these, and report `count` next to every average so a reader can see how much data it rests on.

1. Mean closing price per ticker, and the number of days each mean is built from.
2. Each position's value on **2026-09-30** — that is `shares × close` — plus the total portfolio value and which holding is the largest.
3. Mean daily volume across all nine rows. Then compute it again with the zero-volume day excluded and report the difference.

That third one is the real question of the day. **Was dropping the zero the right call?** Defend your answer in a sentence.

In [8]:
# TODO: 1 -- mean close and day count per ticker
print(px.groupby('ticker')['close'].agg(['mean', 'count']))

# TODO: 2 -- position values on the last date, total, and the largest
last = px[px['date'] == '2026-09-30'].copy()
last['value'] = last['shares'] * last['close']
print(last[['ticker', 'shares', 'close', 'value']])
print('total:', last['value'].sum())
print('largest:', last.loc[last['value'].idxmax(), 'ticker'])

# TODO: 3 -- mean volume with and without the zero day
all_days = px['volume'].agg(['mean', 'count'])
no_zero  = px.loc[px['volume'] != 0, 'volume'].agg(['mean', 'count'])
print(all_days, no_zero, 'difference:', no_zero['mean'] - all_days['mean'])

           mean  count
ticker                
AAPL    231.450      3
KO       71.075      2
MSFT    502.050      3
  ticker  shares  close    value
2   AAPL     120  233.1  27972.0
5   MSFT      45  505.3  22738.5
8     KO     300   70.9  21270.0
total: 71980.5
largest: AAPL
mean     2.667778e+07
count    9.000000e+00
Name: volume, dtype: float64 mean     30012500.0
count           8.0
Name: volume, dtype: float64 difference: 3334722.222222224


**TODO:** the zero-volume question: I kept the zero in the main mean because I'm treating it as a recorded value, not a missing one. If I dropped it, I'd be treating a measurement like a null. I did report the no-zero mean next to it and flagged the row, since MSFT trading zero shares on a normal weekday is implausible and the same record also had a deformed string close (which could suggest a bad record... worth confirming with the source).

### Step 8 — write the result back out

Your output is somebody else's input, so send the context with it — the date and the currency, not a bare table of numbers.

**TODO:** write your per-ticker summary to `portfolio_summary.json` with `indent=2` and `allow_nan=False`, then print the file back. If `allow_nan=False` raises, that is the point of it — a `NaN` reached your output and you have to decide what it should say before it leaves your machine.

In [9]:
# TODO: build the result dict, json.dump it, and print the file back
summary = px.groupby('ticker')['close'].agg(['mean', 'count'])
result = {
    'as_of': data['as_of'],
    'currency': data['currency'],
    'portfolio': data['portfolio']['name'],
    'tickers': [{'ticker': t, 'mean_close': round(float(r['mean']), 2), 'days': int(r['count'])}
                for t, r in summary.iterrows()],
}
with open('portfolio_summary.json', 'w') as f:
    json.dump(result, f, indent=2, allow_nan=False)
print(open('portfolio_summary.json').read())

{
  "as_of": "2026-09-30",
  "currency": "USD",
  "portfolio": "Cavalier Growth",
  "tickers": [
    {
      "ticker": "AAPL",
      "mean_close": 231.45,
      "days": 3
    },
    {
      "ticker": "KO",
      "mean_close": 71.08,
      "days": 2
    },
    {
      "ticker": "MSFT",
      "mean_close": 502.05,
      "days": 3
    }
  ]
}


---

## Part 2 — a second payload, if you have time

This one is already formatted, and it is deeper: a game-day response nested three levels down. Nothing here is new — it is the same `json_normalize` from Step 5 reaching further down, then the `merge` you have been using since Sep 14. The only twist is that you pull **two** tables out of one payload and join them back together.

In [10]:
response = {
    'game': 'UVA vs Duke',
    'date': '2026-10-10',
    'attendance': 47200,
    'zones': [
        {'zone': 'A', 'capacity': 400, 'vendors': [
            {'vendor_id': 'V-01', 'name': 'Hoos Burgers',
             'sales': {'orders': 300, 'revenue': 2250.0},
             'lines': [{'item': 'Cheeseburger', 'qty': 180},
                       {'item': 'Hot Dog', 'qty': 120}]},
            {'vendor_id': 'V-07', 'name': 'Stadium Dogs',
             'sales': {'orders': 210},
             'lines': [{'item': 'Hot Dog', 'qty': 210}]},
        ]},
        {'zone': 'B', 'capacity': 150, 'vendors': [
            {'vendor_id': 'V-12', 'name': 'Wahoo Wings',
             'sales': {'orders': 150, 'revenue': 1800.0},
             'lines': [{'item': 'Wings 6pc', 'qty': 90},
                       {'item': 'Wings 12pc', 'qty': 60}]},
        ]},
        {'zone': 'C', 'capacity': 120, 'vendors': [
            {'vendor_id': 'V-18', 'name': 'Rally Rain Gear',
             'sales': {'orders': 95, 'revenue': 570.0},
             'lines': [{'item': 'Rain Poncho', 'qty': 95}]},
        ]},
    ],
}
print('vendors:', sum(len(z['vendors']) for z in response['zones']))
print('line items:', sum(len(v['lines'])
                         for z in response['zones'] for v in z['vendors']))

vendors: 4
line items: 6


Four vendors, six line items.

**TODO A.** Flatten to one row per vendor, carrying down the game, the date, and the zone's `zone` and `capacity`. Use `sep='_'` and leave the column names it gives you alone. *4 rows.*

**TODO B.** One vendor is missing a field. Find it, and add a boolean column recording whether revenue was reported. Do **not** fill it with zero — write a comment saying why that would be wrong here.

**TODO C.** Flatten the same payload again, down to line items this time, keeping `vendor_id` and `zone`. *6 rows, quantities summing to 755.*

**TODO D.** Join the two, so each line item carries its vendor name and zone capacity. The vendor ID is not spelled the same way in both tables, so reach for `left_on` and `right_on` the way you did on Sep 16. Use `indicator=True` and confirm you still have 6 rows with nothing unmatched.

In [11]:
# TODO A, B, C, D
# A
vendors = pd.json_normalize(response, record_path=['zones', 'vendors'],
                            meta=['game', 'date', ['zones', 'zone'], ['zones', 'capacity']],
                            sep='_')
assert len(vendors) == 4

# B
vendors['revenue_reported'] = vendors['sales_revenue'].notna()


# C
lines = pd.json_normalize(response, record_path=['zones', 'vendors', 'lines'],
                          meta=[['zones', 'zone'], ['zones', 'vendors', 'vendor_id']],
                          sep='_')
assert len(lines) == 6 and lines['qty'].sum() == 755

# D
joined = lines.merge(vendors[['vendor_id', 'name', 'zones_capacity']],
                     left_on='zones_vendors_vendor_id', right_on='vendor_id',
                     how='left', indicator=True)
assert len(joined) == 6
print(joined['_merge'].value_counts())

_merge
both          6
left_only     0
right_only    0
Name: count, dtype: int64


---

## Checkpoint (participation)

Report your row counts from the portfolio payload and what you decided about the zero-volume day.

Work in groups if you wish, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Thursday 11:59pm ET**. One submission per person, not per group.

In [12]:
# Checkpoint
holdings_count = 3       # TODO: should be 3
price_rows = 9           # TODO: should be 9
largest_position = 'AAPL'   # TODO: which ticker, on 2026-09-30
null_vs_zero = "KO's None close on 09-29 was missing data, MSFT's 0 volume on 09-29 was a recorded measurement"       # TODO: which value was missing, which was a real measurement
zero_volume_call = "Kept it in the mean because it's a recorded value, but reported the no-zero mean alongside and flagged it as suspicious"   # TODO: did you keep the zero in the mean, and why

print('holdings:', holdings_count)
print('price rows:', price_rows)
print('largest position:', largest_position)
print('null vs zero:', null_vs_zero)
print('zero volume:', zero_volume_call)

holdings: 3
price rows: 9
largest position: AAPL
null vs zero: KO's None close on 09-29 was missing data, MSFT's 0 volume on 09-29 was a recorded measurement
zero volume: Kept it in the mean because it's a recorded value, but reported the no-zero mean alongside and flagged it as suspicious
